# Forecast tomorrow's regional consumption: production notebook

This notebook is the production job, and only the production job: it retrains one model per region on the most recent 730 days
and writes the forecast of every hour of tomorrow (Paris time) to `regional_forecasts`.
The exploration and the choice of the model live in `eco2mix_regional_exploration.ipynb`.

Run it once from top to bottom to check it, then export it as a **Custom PySpark** action with the Data Platform extension.
Every code cell becomes part of the job: keep checks and explorations out of it.

`holidays` and `scikit-learn` come from the **Dependencies** of the notebook server.

Kernel: **Python 3 (ipykernel)**.

## 1. Set up

Messages go through `logging`: they show below each cell in the notebook, and in Logs Explorer once the notebook runs as an action.

In [ ]:
import logging
from datetime import datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo

import holidays
from forepaas.dwh import connect
from forepaas.dwh.lineage import lineage_run
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

logger = logging.getLogger(__name__)

PARIS = ZoneInfo("Europe/Paris")
FMT = "%Y-%m-%d %H:%M:%S"
TRAIN_DAYS = 730

FEATURE_CATEGORIES = ["hour_day", "hour_lag_day", "month_name"]
FEATURE_NUMBERS = ["lag_48h", "lag_168h"]
MODEL_COLUMNS = ["region_code", "hour_utc", "split", "consumption_mw"] + FEATURE_CATEGORIES + FEATURE_NUMBERS
PREDICTION_SCHEMA = "region_code long, hour_utc timestamp_ntz, predicted_mw double, train_hours long"

In [ ]:
logging.basicConfig(level=logging.INFO)

# connect() first: in the notebook it creates the Spark session with the lakehouse settings,
# which views need; in the action the session already exists. getOrCreate returns it
source = connect("dwh/default_dataset/")
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

spark.sql(f"USE {source.catalog}.{source.schema}")
spark.conf.set("spark.sql.session.timeZone", "UTC")
# 12 regions: a few shuffle partitions are enough, instead of the default 200
spark.conf.set("spark.sql.shuffle.partitions", "12")
logger.info(f"Spark {spark.version}, catalog {source.catalog}")

## 2. Features

Calendar in Paris time, public holidays counted as Sundays, and the consumption 48 and 168 hours earlier.
On the day clocks go back, one UTC hour is missing: it takes the consumption of the hour before.

In [ ]:
def with_paris_time(df):
    return (
        df.withColumn("hour_paris", F.from_utc_timestamp(F.col("hour_utc").cast("timestamp"), "Europe/Paris"))
        .withColumn("hour_of_day", F.hour("hour_paris"))
        .withColumn("year", F.year("hour_paris"))
        .withColumn("month", F.month("hour_paris"))
    )


def holiday_table(spark):
    return spark.createDataFrame(
        [(day, name) for day, name in sorted(holidays.France(years=range(2012, 2030)).items())],
        ["date_paris", "holiday"],
    )


def with_calendar(df, holiday_dates):
    """Day types in Paris time, public holidays counted as Sundays (6)."""
    hol = F.broadcast(holiday_dates.select("date_paris", "holiday", F.lit(1).alias("is_holiday")))
    lag_hol = F.broadcast(holiday_dates.select(F.col("date_paris").alias("lag_date"), F.lit(1).alias("lag_is_holiday")))
    lag_time = F.col("hour_paris") - F.expr("INTERVAL 48 HOURS")
    return (
        with_paris_time(df)
        .withColumn("date_paris", F.to_date("hour_paris"))
        .withColumn("lag_date", F.to_date(lag_time))
        .join(hol, "date_paris", "left")
        .join(lag_hol, "lag_date", "left")
        .withColumn("day_type", F.when(F.col("is_holiday") == 1, 6).otherwise(F.weekday("hour_paris")))
        .withColumn("lag_day_type", F.when(F.col("lag_is_holiday") == 1, 6).otherwise(F.weekday(lag_time)))
        .withColumn("hour_day", F.concat_ws("_", F.col("hour_of_day").cast("string"), F.col("day_type").cast("string")))
        .withColumn("hour_lag_day", F.concat_ws("_", F.col("hour_of_day").cast("string"), F.col("lag_day_type").cast("string")))
        .drop("is_holiday", "lag_is_holiday", "lag_date")
    )


def with_lags(df, hourly):
    """Join each region's consumption 48 and 168 hours earlier, on region and time.

    The dataset follows Paris time: on the day clocks go back, one UTC hour is missing.
    A missing hour takes the consumption of the hour before.
    """
    for hours in (48, 168):
        for shift in (0, 1):
            lag = hourly.select(
                "region_code",
                (F.col("hour_utc") + F.expr(f"INTERVAL {hours + shift} HOURS")).alias("hour_utc"),
                F.col("consumption_mw").alias(f"lag_{hours}h_{shift}"),
            )
            df = df.join(lag, ["region_code", "hour_utc"], "left")
        df = df.withColumn(
            f"lag_{hours}h", F.coalesce(f"lag_{hours}h_0", f"lag_{hours}h_1")
        ).drop(f"lag_{hours}h_0", f"lag_{hours}h_1")
    return df


def prepare(df, split):
    return df.withColumn("split", F.lit(split)).withColumn("month_name", F.col("month").cast("string")).select(MODEL_COLUMNS)

## 3. One model per region

`applyInPandas` runs this function once per region, on a pandas DataFrame holding all the rows of that region.

In [ ]:
def fit_and_forecast(rows):
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    train = rows[rows["split"] == "train"]
    target = rows[rows["split"] == "target"]
    if train.empty or target.empty:
        return pd.DataFrame(columns=["region_code", "hour_utc", "predicted_mw", "train_hours"])

    model = Pipeline([
        ("encode", ColumnTransformer(
            [("calendar", OneHotEncoder(handle_unknown="ignore"), FEATURE_CATEGORIES)],
            remainder="passthrough",
        )),
        ("regress", LinearRegression()),
    ])
    model.fit(train[FEATURE_CATEGORIES + FEATURE_NUMBERS], train["consumption_mw"])

    forecast = target[["region_code", "hour_utc"]].copy()
    forecast["predicted_mw"] = model.predict(target[FEATURE_CATEGORIES + FEATURE_NUMBERS])
    forecast["train_hours"] = len(train)
    return forecast

## 4. Forecast tomorrow and write it

The job raises an error if the lags of tomorrow are missing: run the collection workflow first if the data is not fresh.
It replaces the forecasts of the target day, so a second run on the same day does not duplicate rows.

In [ ]:
def target_window(run_date):
    """First and last-plus-one UTC hours of the day after run_date, in Paris time."""
    day = run_date + timedelta(days=1)
    start = datetime.combine(day, time(0), PARIS).astimezone(timezone.utc)
    end = datetime.combine(day + timedelta(days=1), time(0), PARIS).astimezone(timezone.utc)
    return start.strftime(FMT), end.strftime(FMT)


def forecast_tomorrow(spark, run_date=None, train_days=TRAIN_DAYS):
    """Retrain each region on its most recent hours and forecast every hour of the next day."""
    if run_date is None:
        run_date = datetime.now(PARIS).date()
    start, end = target_window(run_date)
    run_at = datetime.now(timezone.utc).strftime(FMT)

    holiday_dates = holiday_table(spark)
    view = spark.table("regional_consumption_hourly")

    # Latest measured hour, read from the small real time table: reading it from the view
    # would aggregate the whole history first
    latest_quarter = (
        spark.table("eco2mix_regional_realtime")
        .where(F.col("consommation").isNotNull())
        .agg(F.max("date_heure"))
        .first()[0]
    )
    last_measured = latest_quarter.replace(minute=0, second=0, microsecond=0)

    # Keep only the hours the job needs: the training window, plus 8 days so that the oldest rows still get their 168 h lag
    hourly = view.where(
        F.col("hour_utc") > F.lit(last_measured) - F.expr(f"INTERVAL {train_days + 8} DAYS")
    ).cache()

    history = (
        with_lags(with_calendar(hourly, holiday_dates), hourly)
        .where(F.col("consumption_mw").isNotNull())
        .dropna(subset=["lag_48h", "lag_168h"])
    )
    train = history.where(F.col("hour_utc") > F.lit(last_measured) - F.expr(f"INTERVAL {train_days} DAYS"))

    regions = hourly.select("region_code", "region_name").distinct()
    target_hours = spark.range(1).select(
        F.explode(F.sequence(
            F.lit(start).cast("timestamp_ntz"),
            F.lit(end).cast("timestamp_ntz") - F.expr("INTERVAL 1 HOUR"),
            F.expr("INTERVAL 1 HOUR"),
        )).alias("hour_utc")
    )
    target = with_lags(with_calendar(regions.crossJoin(target_hours), holiday_dates), hourly).withColumn(
        "consumption_mw", F.lit(None).cast("double")
    )

    missing = target.where(F.col("lag_48h").isNull() | F.col("lag_168h").isNull()).count()
    if missing:
        raise ValueError(
            f"{missing} region-hours of {run_date + timedelta(days=1)} have no lag: "
            f"the latest measured hour is {last_measured}, collect eco2mix_regional_realtime first"
        )

    forecast = (
        prepare(train, "train").unionByName(prepare(target, "target"))
        .groupBy("region_code")
        .applyInPandas(fit_and_forecast, schema=PREDICTION_SCHEMA)
        .join(regions, "region_code")
        .select(
            F.col("hour_utc").alias("forecast_hour"),
            "region_code",
            "region_name",
            "predicted_mw",
            F.lit(run_at).cast("timestamp_ntz").alias("run_at"),
            "train_hours",
        )
    )
    return forecast, start, end, last_measured


def write_forecast(spark, source, forecast, start, end):
    """Replace the forecasts of the target day, then append the new ones."""
    spark.sql(
        f"DELETE FROM regional_forecasts "
        f"WHERE forecast_hour >= TIMESTAMP_NTZ '{start}' AND forecast_hour < TIMESTAMP_NTZ '{end}'"
    )
    source.insert_dataframe(table="regional_forecasts", df=forecast.drop("train_hours"), mode="append")

## 5. Run

`lineage_run` declares what the job reads and writes, `setJobDescription` names its Spark jobs in the Spark History Server.

In [ ]:
# Lineage is opt-in for Custom actions: declare what the job reads and writes
with lineage_run(
    "regional_forecast",
    inputs=[
        "default_dataset/eco2mix_regional_history",
        "default_dataset/eco2mix_regional_realtime",
    ],
    outputs=["default_dataset/regional_forecasts"],
    connector=source,
):
    sc.setJobDescription("Train one model per region and forecast tomorrow")
    forecast, start, end, last_measured = forecast_tomorrow(spark)
    forecast = forecast.cache()

    rows = forecast.count()
    logger.info(f"Latest measured hour: {last_measured}")
    logger.info(f"Forecast {start} to {end} (UTC): {rows} region-hours")
    for r in forecast.groupBy("region_name").agg(F.first("train_hours").alias("train_hours")).orderBy("region_name").collect():
        logger.info(f"{r['region_name']}: trained on {r['train_hours']} hours")

    sc.setJobDescription("Write regional_forecasts")
    write_forecast(spark, source, forecast, start, end)
    logger.info("regional_forecasts written")

To check the result, run in the Lakehouse Manager Explorer:

```sql
SELECT run_at, count(*) AS forecasts
FROM regional_forecasts
GROUP BY run_at
ORDER BY run_at
```

Run the notebook a second time: `regional_forecasts` must still hold 288 forecasts for tomorrow, with a newer `run_at`.